In [0]:
from pyspark.sql import SparkSession, functions as F
import builtins, sys

# Ensure spark is accessible by module-level code in edtech_gold_utils
builtins.spark = SparkSession.builder.getOrCreate()
sys.modules.pop("edtech_gold_utils", None)

from edtech_gold_utils import (
    TABLES,
    require_table,
    has_col,
    col_or_null,
    pct_norm,
    overwrite_delta_table,
)

In [0]:
from edtech_gold_utils import gold_schema

require_table(TABLES['engagement'])
enr = spark.table(TABLES['engagement'])

enr_base = enr.select(
    F.col("student_id").alias("student_id"),
    F.col("course_id").alias("course_id"),
    col_or_null(enr, "enrolled_date", "date").alias("enrolled_date"),
    col_or_null(enr, "status", "string").alias("status"),
    (
        F.to_date("last_accessed_date")
        if has_col(enr, "last_accessed_date")
        else F.lit(None).cast("date")
    ).alias("last_accessed_date"),
    col_or_null(enr, "current_progress_pct", "double").alias("current_progress_pct"),
)

enr_base = (
    enr_base
    .withColumn(
        "started_flag",
        F.when(
            (F.col("last_accessed_date").isNotNull() & (F.col("last_accessed_date") >= F.col("enrolled_date"))) |
            (F.coalesce(F.col("current_progress_pct"), F.lit(0.0)) > 0),
            1
        ).otherwise(0)
    )
    .withColumn("q25_flag", F.when(F.coalesce(F.col("current_progress_pct"), F.lit(0.0)) >= 25, 1).otherwise(0))
    .withColumn("q50_flag", F.when(F.coalesce(F.col("current_progress_pct"), F.lit(0.0)) >= 50, 1).otherwise(0))
    .withColumn("q75_flag", F.when(F.coalesce(F.col("current_progress_pct"), F.lit(0.0)) >= 75, 1).otherwise(0))
    .withColumn(
        "completed_flag",
        F.when(
            (F.coalesce(F.col("current_progress_pct"), F.lit(0.0)) >= 100) |
            (F.upper(F.col("status")) == F.lit("COMPLETED")),
            1
        ).otherwise(0)
    )
    .withColumn(
        "cohort_quarter",
        F.concat(
            F.year("enrolled_date").cast("string"),
            F.lit("-Q"),
            F.quarter("enrolled_date").cast("string")
        )
    )
)

def build_funnel(df, group_cols, analysis_scope_value):
    base = (
        df.groupBy(*group_cols)
        .agg(
            F.count("*").alias("enrolled_count"),
            (F.sum("started_flag") * 100.0 / F.count("*")).alias("started_pct"),
            (F.sum("q25_flag") * 100.0 / F.count("*")).alias("q25_pct"),
            (F.sum("q50_flag") * 100.0 / F.count("*")).alias("q50_pct"),
            (F.sum("q75_flag") * 100.0 / F.count("*")).alias("q75_pct"),
            (F.sum("completed_flag") * 100.0 / F.count("*")).alias("completed_pct"),
        )
    )

    base = (
        base
        .withColumn("dropoff_enrolled_started", 100.0 - F.col("started_pct"))
        .withColumn("dropoff_started_q25", F.col("started_pct") - F.col("q25_pct"))
        .withColumn("dropoff_q25_q50", F.col("q25_pct") - F.col("q50_pct"))
        .withColumn("dropoff_q50_q75", F.col("q50_pct") - F.col("q75_pct"))
        .withColumn("dropoff_q75_completed", F.col("q75_pct") - F.col("completed_pct"))
        .withColumn(
            "biggest_dropoff_stage",
            F.when(
                F.col("dropoff_enrolled_started") >= F.greatest(
                    "dropoff_started_q25", "dropoff_q25_q50", "dropoff_q50_q75", "dropoff_q75_completed"
                ),
                F.lit("ENROLLED->STARTED")
            )
            .when(
                F.col("dropoff_started_q25") >= F.greatest(
                    "dropoff_enrolled_started", "dropoff_q25_q50", "dropoff_q50_q75", "dropoff_q75_completed"
                ),
                F.lit("STARTED->25%")
            )
            .when(
                F.col("dropoff_q25_q50") >= F.greatest(
                    "dropoff_enrolled_started", "dropoff_started_q25", "dropoff_q50_q75", "dropoff_q75_completed"
                ),
                F.lit("25%->50%")
            )
            .when(
                F.col("dropoff_q50_q75") >= F.greatest(
                    "dropoff_enrolled_started", "dropoff_started_q25", "dropoff_q25_q50", "dropoff_q75_completed"
                ),
                F.lit("50%->75%")
            )
            .otherwise(F.lit("75%->COMPLETED"))
        )
        .withColumn("analysis_scope", F.lit(analysis_scope_value))
    )
    return base

# Overall course funnel
overall_funnel = (
    build_funnel(enr_base, ["course_id"], "COURSE_OVERALL")
    .withColumn("cohort_quarter", F.lit("ALL"))
    .select(
        "analysis_scope",
        "course_id",
        "cohort_quarter",
        "enrolled_count",
        "started_pct",
        "q25_pct",
        "q50_pct",
        "q75_pct",
        "completed_pct",
        "biggest_dropoff_stage",
    )
)

# Cohort funnel by quarter so Q1 vs Q2 comparison is possible directly in the same table
cohort_funnel = (
    build_funnel(enr_base, ["course_id", "cohort_quarter"], "COHORT_ANALYSIS")
    .select(
        "analysis_scope",
        "course_id",
        "cohort_quarter",
        "enrolled_count",
        "started_pct",
        "q25_pct",
        "q50_pct",
        "q75_pct",
        "completed_pct",
        "biggest_dropoff_stage",
    )
)

course_completion_funnel = overall_funnel.unionByName(cohort_funnel)

overwrite_delta_table(
    course_completion_funnel,
    f"{gold_schema}.course_completion_funnel",
    partition_cols=["analysis_scope", "cohort_quarter"],
)

In [0]:
final_df = spark.sql(f"""
SELECT * FROM {gold_schema}.course_completion_funnel
""")
display(final_df)